# Recomendador explicable: entrenamiento y visualizaciones

Carga un conjunto de datos, entrena el modelo sobre la partición de entrenamiento,
recomienda y genera las figuras de las dos capas de visualización.

Las tablas de la tesis se reproducen con los scripts de `analisis/` (ver `README.md`).

## 1. Preparación

In [ ]:
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
RAIZ = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content")]
             if (p / "evolutionary_cf.py").exists()), None)
if RAIZ is None and EN_COLAB:
    from google.colab import files
    print("Sube evolutionary_cf.py y explanation_metrics.py")
    files.upload()
    RAIZ = Path("/content")
if RAIZ is None:
    raise SystemExit("Ejecuta el cuaderno desde la carpeta que contiene evolutionary_cf.py")
sys.path.insert(0, str(RAIZ))

try:
    import matplotlib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

import evolutionary_cf as ecf
import explanation_metrics as em
from evolutionary_cf import (
    EvolutionaryCollaborativeFilter, known_items_by_user, load_bookcrossing_folder,
    load_lastfm_hetrec_folder, load_movielens_100k, generate_explanation_plots,
    generate_user_explanation_plots,
)
print("módulos en", RAIZ)

## 2. Datos

MovieLens 100K y Last.fm HetRec2011 se descargan de GroupLens si no están en `datasets/`.
Book-Crossing debe colocarse a mano (`Ratings.csv` y `Books.csv`).

In [ ]:
import io, urllib.request, zipfile

DATOS = RAIZ / "datasets"


def descargar(url, destino):
    destino.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(destino)


if not (DATOS / "ml-100k" / "u.data").exists():
    descargar("https://files.grouplens.org/datasets/movielens/ml-100k.zip", DATOS)
if not (DATOS / "hetrec-lastfm" / "user_artists.dat").exists():
    descargar("https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip",
              DATOS / "hetrec-lastfm")

RUTAS = {
    "movielens": DATOS / "ml-100k",
    "lastfm": DATOS / "hetrec-lastfm",
    "bookcrossing": DATOS if (DATOS / "Ratings.csv").exists() else None,
}
for nombre, ruta in RUTAS.items():
    print(f"{nombre:13s} {ruta if ruta else 'no disponible'}")

## 3. Configuración

In [ ]:
DATASET = "movielens"      # MOVIELENS BOOKCROSSING O LASTFM
USER_ID = None             # NONE USA EL CASO DE ESTUDIO DE LA TESIS
TOP_N = 10
RAPIDO = True              # FALSE USA LA CONFIGURACIÓN COMPLETA DE LA TESIS
SEED = 42

CONFIG = dict(factors=6, population_size=25, generations=120, mutation_sigma=0.08,
              mutation_rate=0.02, crossover_rate=0.60, guidance_rate=0.01,
              guidance_batch=8192, init_scale=0.3, regularization=0.001)
if RAPIDO:
    CONFIG.update(population_size=10, generations=25, guidance_batch=4096)

CASOS = {"movielens": "224", "bookcrossing": "3331", "lastfm": "333"}
UMBRAL_RELEVANCIA = {"movielens": 4.0, "bookcrossing": 4.0, "lastfm": 3.0}

## 4. Carga y partición

In [ ]:
ruta = RUTAS[DATASET]
if ruta is None:
    raise SystemExit(f"Faltan los archivos de {DATASET} en {DATOS}")

if DATASET == "movielens":
    ds = load_movielens_100k(ruta)
elif DATASET == "bookcrossing":
    ds = load_bookcrossing_folder(ruta, 5, 5)
else:
    ds = load_lastfm_hetrec_folder(ruta, 5, 5)

split = em.split_train_test_by_user(ds.ratings, 0.20, SEED)
train_ds = em.dataset_with_ratings(ds, split.train)
conocidos = known_items_by_user(train_ds.ratings)

print(f"usuarios {len(ds.idx_to_user):,}  ítems {len(ds.idx_to_item):,}  "
      f"interacciones {len(ds.ratings):,}")
print(f"entrenamiento {len(split.train):,}  prueba {len(split.test):,}")

## 5. Entrenamiento

In [ ]:
import time

t0 = time.time()
model = EvolutionaryCollaborativeFilter(
    n_users=len(ds.idx_to_user), n_items=len(ds.idx_to_item),
    seed=SEED, verbose=True, **CONFIG,
).fit(split.train)

usuarios = em.sample_user_indices(ds, len(ds.idx_to_user), TOP_N, SEED, train_ratings=split.train)
calidad = em.evaluate_recommender_quality(model, train_ds, split.test, usuarios, TOP_N,
                                          UMBRAL_RELEVANCIA[DATASET])
print(f"\nentrenado en {time.time() - t0:.0f} s")
print(f"fitness {model.best_.fitness:.4f}  RMSE de prueba {calidad['rmse']:.4f}  "
      f"NDCG@{TOP_N} {calidad[f'ndcg_at_{TOP_N}']:.4f}")

## 6. Recomendaciones

In [ ]:
user_id = str(USER_ID or CASOS[DATASET])
user_idx = ds.user_to_idx[user_id]
recs, corte = model.recommend_with_cutoff(user_idx, conocidos.get(user_idx, set()), TOP_N)

print(f"usuario {user_id}: {len(conocidos.get(user_idx, set()))} interacciones de entrenamiento\n")
for n, (item_idx, score) in enumerate(recs, 1):
    print(f"{n:2d}. {ecf.item_label(ds, item_idx)[:60]:<60} {score:.3f}")

## 7. Descomposición y evidencia de la primera recomendación

In [ ]:
item_idx, _ = recs[0]
comp = model.prediction_components(user_idx, item_idx)
evidencia = ecf.build_recommendation_evidence(model, train_ds, user_idx, recs)
fila = em.evaluate_user_recommendations(model, user_idx, recs, evidencia,
                                        next_best_raw_score=corte)[0]

print(ecf.item_label(ds, item_idx))
print(f"  media global       {comp['global_mean']:+.4f}")
print(f"  sesgo de usuario   {comp['user_bias']:+.4f}")
print(f"  sesgo de ítem      {comp['item_bias']:+.4f}")
print(f"  afinidad latente   {comp['latent_dot']:+.4f}")
for j, v in enumerate(comp["factor_contributions"], 1):
    print(f"      F{j}  {v:+.4f}")
print(f"  predicción         {comp['prediction']:+.4f}")
print(f"\n  confianza {evidencia[0]['confidence']:.4f}  "
      f"ablation shift {fila['ablation_shift']:.4f}  "
      f"necesidad {bool(fila['necessity'])}  suficiencia {bool(fila['sufficiency'])}")
print("\n  evidencia:")
for i, sim, rating in evidencia[0]["similar_known"]:
    print(f"    {ecf.item_label(ds, i)[:50]:<50} similitud {sim:+.3f}  valoración {rating:.1f}")

## 8. Capa técnica

In [ ]:
from IPython.display import Image, display

for ruta_fig in generate_explanation_plots(model, train_ds, user_id, recs,
                                           Path("figuras") / DATASET / "tecnicas"):
    print(ruta_fig.name)
    display(Image(filename=str(ruta_fig)))

## 9. Capa de usuario final

In [ ]:
for ruta_fig in generate_user_explanation_plots(model, train_ds, user_id, recs,
                                                Path("figuras") / DATASET / "usuario"):
    print(ruta_fig.name)
    display(Image(filename=str(ruta_fig)))